<a href="https://colab.research.google.com/github/Rubal-code/DEEP-LEARNING/blob/main/fine_tuning_llama_with_unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# unsloth is the famous lib for fine tune the llm models i will use t4 gpu

In [2]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

In [3]:
import torch
from unsloth import FastLanguageModel

In [4]:
max_seq_length = 2048
# Maximum number of tokens the model can handle at once

dtype = None
# Automatically choose the best data type for your GPU

load_in_4bit = True
# Load the model in 4-bit to save GPU memory
# This is used in QLoRA

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    # Which pretrained model to load

    max_seq_length = max_seq_length,
    # Maximum text length = 2048 tokens

    dtype = dtype,
    # Automatically choose the suitable number format

    load_in_4bit = load_in_4bit
    # Load model in 4-bit to save GPU memory
)

==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:42: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,

    r = 16,
    # LoRA rank — controls the size/capacity of LoRA

    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    # These are the parts of the model where LoRA is added

    lora_alpha = 16,
    # Controls the strength of LoRA

    lora_dropout = 0,
    # No dropout → commonly used for efficient fine-tuning

    bias = "none",
    # Do not train bias parameters

    use_gradient_checkpointing = "unsloth",
    # Saves GPU memory during training

    random_state = 3407,
    # Makes results more reproducible

    use_rslora = False,
    # Use normal LoRA scaling

    loftq_config = None,
    # No LoftQ initialization
)

Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [8]:
from datasets import load_dataset
dataset = load_dataset("Abirate/english_quotes",split='train')

quotes.jsonl: reconstructing file:   0%|          |  0.00B /  647kB            

quotes.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [9]:
dataset[:2]

{'quote': ['“Be yourself; everyone else is already taken.”',
  "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”"],
 'author': ['Oscar Wilde', 'Marilyn Monroe'],
 'tags': [['be-yourself',
   'gilbert-perreira',
   'honesty',
   'inspirational',
   'misattributed-oscar-wilde',
   'quote-investigator'],
  ['best', 'life', 'love', 'mistakes', 'out-of-control', 'truth', 'worst']]}

In [11]:
# New Prompt format for the English Quotes dataset
# We will instruct the model to generate a quote by a given author.
prompt_template = """### Instruction:
You are an expert in quotes. Generate a well-known quote by the author {author}.

### Response:
"{quote}"
"""

# Get the End-Of-Sequence token from the tokenizer
# This tells the model that one training example has finished
EOS_TOKEN = tokenizer.eos_token


# Function to format the dataset
def formatting_prompts_func(examples):

    # Get the quote and author columns
    quotes = examples["quote"]
    authors = examples["author"]

    # Empty list to store formatted examples
    texts = []

    # Take one quote and author at a time
    for quote, author in zip(quotes, authors):

        # Put author and quote inside our prompt template
        text = prompt_template.format(
            author=author,
            quote=quote
        ) + EOS_TOKEN

        # Add completed example to the list
        texts.append(text)

    # Return formatted examples in a new "text" column
    return {"text": texts}


# Apply the formatting function to the whole dataset
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [14]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2508 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,508 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWar

Step,Training Loss
1,3.393550
2,3.496411
3,2.792192
4,3.282825
5,3.210788
6,3.148749
7,3.110016
8,2.327649
9,1.531007
10,2.175282


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TrainOutput(global_step=60, training_loss=1.6038096616665523, metrics={'train_runtime': 134.3718, 'train_samples_per_second': 3.572, 'train_steps_per_second': 0.447, 'total_flos': 666026900201472.0, 'train_loss': 1.6038096616665523, 'epoch': 0.19138755980861244})

## Inference
Let's test the fine-tuned model with a new author.

In [16]:
# We will now test the model to generate a quote for a new author.

# Prepare the input prompt for inference
# Create a specific prompt for inference that doesn't include the empty quote placeholder.
author_for_inference = "William Shakespeare"
inference_instruction = f"### Instruction:\nYou are an expert in quotes. Generate a well-known quote by the author {author_for_inference}.\n\n### Response:\n"

# Encode the prompt and generate a response
inputs = tokenizer([inference_instruction], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 64,
    use_cache = True,
    pad_token_id = tokenizer.eos_token_id,
)

# Decode and print the generated text
text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Extract just the generated quote response part
# This assumes the model follows the prompt format well
response_start = text.find("### Response:\n")
if response_start != -1:
    generated_text = text[response_start + len("### Response:\n"):].strip()
    print(generated_text)
else:
    print("Generated text does not contain '### Response:', printing full output:\n", text)

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"“Love is merely a madness. And maddest of all, to love oneself.”"
